# Minimale Komponentenlieferkette für Windkraft

Dieses Beispiel interpretiert `materialIntensity` als **Komponentenintensität**. Eine endogen gebaute Windkraftanlage benötigt Leitschaufeln. Diese werden in einer `Conversion` aus zwei Rohmaterialien hergestellt.

Das Modell hat bewusst nur eine Region und ein vollständiges Jahr mit 8.760 stündlichen Zeitschritten. Dadurch wird eine wichtige Eigenschaft sichtbar: Die einmalige, investitionsabhängige Nachfrage nach Leitschaufeln ist nur als Jahressumme festgelegt. Die Fertigungs-Conversion darf diese Produktion frei auf die 8.760 Stunden verteilen.

In [1]:
import contextlib
import io

import fine as fn
import numpy as np
import pandas as pd
import pyomo.environ as pyomo

## 1. Modellannahmen

- konstante Stromnachfrage: 0,5 MW in jeder Stunde
- konstantes Windprofil: 50 % in jeder Stunde
- daraus folgt eine benötigte Windkapazität von 1 MW
- Komponentenintensität: 4 Leitschaufeln je MW neuer Windkapazität
- Stückliste je Leitschaufel: 2 Einheiten Material X und 3 Einheiten Material Y

Die Sources von X und Y stehen für eine exogene Bereitstellung am Rand des modellierten Systems. Sie bilden keine eigene Produktionstechnologie ab.

In [2]:
loc = "Region"
year = 2020
number_of_time_steps = 8760

wind_capacity_factor = 0.5
electricity_demand = 0.5  # MW in every hour
guide_vanes_per_mw = 4.0
material_x_per_vane = 2.0
material_y_per_vane = 3.0

In [3]:
esM = fn.EnergySystemModel(
    locations={loc},
    commodities={"electricity", "material_x", "material_y"},
    commodityUnitsDict={
        "electricity": "MW",
        "material_x": "unit_X",
        "material_y": "unit_Y",
    },
    materials={"guide_vane"},
    materialUnitsDict={"guide_vane": "piece"},
    numberOfTimeSteps=number_of_time_steps,
    hoursPerTimeStep=1,
    numberOfInvestmentPeriods=1,
    investmentPeriodInterval=1,
    startYear=year,
    costUnit="1 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

In [4]:
hourly_wind_profile = pd.DataFrame(
    wind_capacity_factor, index=range(number_of_time_steps), columns=[loc]
)
hourly_electricity_demand = pd.DataFrame(
    electricity_demand, index=range(number_of_time_steps), columns=[loc]
)

## 2. Exogene Rohmaterialbereitstellung

Wie die Steel Source in Beispiel 10c stellen diese Komponenten Material aus einem nicht weiter aufgelösten vorgelagerten System bereit. Die positiven variablen Kosten vermeiden unnötige Materialflüsse.

In [5]:
esM.add(
    fn.Source(
        esM=esM,
        name="Material X supply",
        commodity="material_x",
        hasCapacityVariable=False,
        commodityCost=2.0,
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Material Y supply",
        commodity="material_y",
        hasCapacityVariable=False,
        commodityCost=3.0,
    )
)

## 3. Herstellung der Leitschaufeln

Eine Einheit Conversion-Operation produziert eine Leitschaufel und verbraucht die feste Stückliste aus X und Y. FINE verlangt als `physicalUnit` eine registrierte Commodity-Einheit, hier also `piece`. Da die Operation in stündlichen Zeitschritten durch Kapazität × Stunden begrenzt wird, interpretieren wir die optimierte Fertigungskapazität als Stück pro Stunde.

In [6]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Guide vane production",
        physicalUnit="piece",
        commodityConversionFactors={
            "guide_vane": 1.0,
            "material_x": -material_x_per_vane,
            "material_y": -material_y_per_vane,
        },
        hasCapacityVariable=True,
        investPerCapacity=1.0,
        economicLifetime=20,
        technicalLifetime=20,
    )
)

## 4. Windkraft und Stromnachfrage

Die Windkapazität ist innerhalb des modellierten Jahres konstant. Ihr Zubau bleibt endogen: Das konstante Windprofil und die konstante Nachfrage erzwingen genau 1 MW. `materialIntensity` koppelt vier Leitschaufeln an jedes neu installierte MW.

In [7]:
esM.add(
    fn.Source(
        esM=esM,
        name="Wind power",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateFix=hourly_wind_profile,
        investPerCapacity=1000.0,
        economicLifetime=20,
        technicalLifetime=20,
        materialIntensity={year: {"guide_vane": pd.Series({loc: guide_vanes_per_mw})}},
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=hourly_electricity_demand,
    )
)

## 5. Investitionsabhängiger Komponentensink

Dieser besondere Sink ist die Brücke zwischen der jährlichen `commissioning`-Variable der Windkraft und der stündlichen Commodity-Bilanz der Leitschaufeln. Seine Jahressumme wird durch FINE auf `commissioning × materialIntensity` festgelegt.

In [8]:
esM.add(
    fn.Sink(
        esM=esM,
        name="Guide vane demand",
        commodity="guide_vane",
        hasCapacityVariable=False,
        material=True,
    )
)

In [9]:
# Die Material-Constraint gibt derzeit sehr lange Pyomo-Ausdrücke aus.
# Um das Notebook lesbar zu halten, wird diese Konsolenausgabe abgefangen.
optimization_log = io.StringIO()
with contextlib.redirect_stdout(optimization_log):
    esM.optimize(solver="glpk")

## 6. Mengenbilanzen auswerten

In [10]:
ip = 0


def annual_source_sink_operation(component_name):
    """Return the summed operation of a Source or Sink over the modeled year."""
    return sum(
        pyomo.value(esM.pyM.op_srcSnk[loc, component_name, ip, p, t])
        * esM.periodOccurrences[ip][p]
        for p, t in esM.pyM.intraYearTimeSet
    )


def annual_conversion_operation(component_name):
    """Return the summed operation of a Conversion over the modeled year."""
    return sum(
        pyomo.value(esM.pyM.op_conv[loc, component_name, ip, p, t])
        * esM.periodOccurrences[ip][p]
        for p, t in esM.pyM.intraYearTimeSet
    )


wind_capacity = pyomo.value(esM.pyM.cap_srcSnk[loc, "Wind power", ip])
wind_commissioning = pyomo.value(esM.pyM.commis_srcSnk[loc, "Wind power", ip])
guide_vane_factory_capacity = pyomo.value(
    esM.pyM.cap_conv[loc, "Guide vane production", ip]
)
guide_vane_production = annual_conversion_operation("Guide vane production")

results = pd.Series(
    {
        "wind_capacity_MW": wind_capacity,
        "wind_commissioning_MW": wind_commissioning,
        "annual_electricity_demand_MWh": annual_source_sink_operation(
            "Electricity demand"
        ),
        "guide_vane_demand_piece": annual_source_sink_operation("Guide vane demand"),
        "guide_vane_production_piece": guide_vane_production,
        "material_x_supply": annual_source_sink_operation("Material X supply"),
        "material_y_supply": annual_source_sink_operation("Material Y supply"),
        "guide_vane_factory_capacity_piece_per_h": (guide_vane_factory_capacity),
    }
)
results

wind_capacity_MW                              1.000000
wind_commissioning_MW                         1.000000
annual_electricity_demand_MWh              4380.000000
guide_vane_demand_piece                       4.000000
guide_vane_production_piece                   4.000000
material_x_supply                             8.000000
material_y_supply                            12.000000
guide_vane_factory_capacity_piece_per_h       0.000457
dtype: float64

In [11]:
expected_wind_capacity = electricity_demand / wind_capacity_factor
expected_guide_vanes = expected_wind_capacity * guide_vanes_per_mw

np.testing.assert_allclose(wind_capacity, expected_wind_capacity, rtol=0, atol=1e-7)
np.testing.assert_allclose(
    wind_commissioning, expected_wind_capacity, rtol=0, atol=1e-7
)
np.testing.assert_allclose(
    results["annual_electricity_demand_MWh"],
    electricity_demand * number_of_time_steps,
    rtol=0,
    atol=1e-6,
)
np.testing.assert_allclose(
    results["guide_vane_demand_piece"], expected_guide_vanes, atol=1e-7
)
np.testing.assert_allclose(guide_vane_production, expected_guide_vanes, atol=1e-7)
np.testing.assert_allclose(
    results["material_x_supply"],
    expected_guide_vanes * material_x_per_vane,
    atol=1e-7,
)
np.testing.assert_allclose(
    results["material_y_supply"],
    expected_guide_vanes * material_y_per_vane,
    atol=1e-7,
)
np.testing.assert_allclose(
    guide_vane_factory_capacity,
    expected_guide_vanes / number_of_time_steps,
    rtol=0,
    atol=1e-7,
)

print("All component-chain balances are consistent.")

All component-chain balances are consistent.


## 7. Was zeigt der 8.760-Stunden-Fall?

Die Windkraft benötigt insgesamt vier Leitschaufeln. Der Material-Sink legt aber nur diese **Jahressumme** fest und besitzt kein Bauzeitprofil. Weil die Fertigungskapazität Kosten verursacht, verteilt das Modell die Produktion gleichmäßig auf alle Stunden und baut nur

`4 pieces / 8760 h = 0.0004566 pieces/h`

Fertigungskapazität. Das ist mathematisch konsistent, aber eine starke Annahme: Komponenten können während des gesamten Investitionsjahres produziert und unmittelbar für den im selben Jahr ausgewiesenen Zubau verwendet werden. Soll der gesamte Windpark zu einem bestimmten Zeitpunkt verfügbar sein, bräuchte die Komponentennachfrage ein zeitliches Bauprofil, eine Produktionsvorlaufzeit oder eine Lagerlogik. Die aktuelle `materialIntensity` bildet diese zeitliche Kausalität nicht ab.